## Today {.smaller}

- **Autoencoders**: compress with no labels, and find the odd one out
- **Variational autoencoders**: compress in a way you can sample from

::: {.notes}
Three content meetings left, and they are the generative ones. Autoencoders
earn their place twice over. On their own they do real work, mostly anomaly
detection, which is the half of today you could ship next week. And the
compressor you see today is the specific piece that makes image diffusion
affordable, which is next week. That is not a general law: the language
models from Week 10 generate text with no compressor anywhere in them.
:::

## What an autoencoder is {.smaller}

An autoencoder is a network trained to reproduce its own input through a
bottleneck too small to pass everything.

[![](images/ae-dog.png){.dw58}](images/ae-dog.png){target="_blank" .zoom}

::: {.notes}
The encoder compresses the input to a code, the decoder rebuilds the input
from that code, and the loss is the difference between what went in and what
came out. No labels anywhere. The bottleneck is the whole design: without it
the network can learn the identity function and learn nothing.
:::

## How small should the bottleneck be {.smaller}

[![](images/ae-complete.png){.dw75}](images/ae-complete.png){target="_blank" .zoom}

::: {.notes}
Under-complete means the code is smaller than the input, which forces the
network to throw something away and keep what matters. Over-complete means it
is as large or larger, and then nothing stops it copying the input straight
through. If you want an over-complete code to be useful you have to constrain
it another way, with noise or with a penalty on the activations.
:::

## Autoencoders and PCA {.smaller}

An autoencoder with linear layers and a squared-error loss spans the same
subspace PCA finds.

[![](images/ae-pca.png){.dw55}](images/ae-pca.png){target="_blank" .zoom}

- Add nonlinear activations and it bends, which PCA cannot

::: {.notes}
This is the sentence that places autoencoders for anyone who has used PCA: it
is nonlinear dimensionality reduction. The handwritten note is the caveat. The
linear autoencoder finds the same subspace but not the same basis, so the axes
are not ordered by variance and are not the principal components.
:::

## What people actually use them for {.smaller}

[![](images/ae-applications.png){.dw58}](images/ae-applications.png){target="_blank" .zoom}

**Segmentation is on that list for its shape, not its training.** A U-Net is
an encoder-decoder, but it learns from labeled masks, so it is not an
autoencoder.

::: {.notes}
The ones that pay are dimensionality reduction, denoising, and anomaly
detection, and we will take those in turn.

Segmentation deserves the caveat because the list is misleading without it.
Nobody trains an autoencoder to segment. What happened is that the
encoder-decoder shape was borrowed, which is the U-Net from Week 7. The live
connection today is the other direction: a masked autoencoder reconstructs
hidden patches with no labels at all, and that pretrained encoder is then
fine-tuned for segmentation or detection. Reconstruction is the pretraining
task, never the segmentation task itself.
:::

## Compression {.smaller}

[![](images/ae-compression.png){.dw75}](images/ae-compression.png){target="_blank" .zoom}

::: {.notes}
Send the code instead of the image, and rebuild it at the other end. This is
not how JPEG works and it is not usually how you should compress a photograph,
because a general codec is better and does not need training. It is a useful
picture because it makes clear that the code has to carry everything the
decoder will need.
:::

## Denoising {.smaller}

A denoising autoencoder is trained with corrupted input and the clean original
as the target.

[![](images/ae-denoise.png){.dw60}](images/ae-denoise.png){target="_blank" .zoom}

::: {.notes}
It cannot copy its input, because its input is wrong. The only way to lower
the loss is to learn what the data normally looks like. Hold on to this one.
Next week's diffusion model is this idea taken to its limit: destroy the data
in a way you control, then train a network to undo it.
:::

## Convolutional autoencoders {.smaller}

[![](images/ae-conv.png){.dw70}](images/ae-conv.png){target="_blank" .zoom}

For images, replace the dense layers with convolutions down and transposed
convolutions back up.

::: {.notes}
Same idea, better inductive bias. A dense layer on a 256 by 256 image has
billions of parameters and no notion that nearby pixels are related. The
convolutional version is the one that gets used, and it is the shape of the
U-Net from Week 7 and the denoiser you will meet next week.
:::

## Inpainting {.smaller}

[![](images/ae-inpaint.png){.dw60}](images/ae-inpaint.png){target="_blank" .zoom}

::: {.notes}
Mask part of the image, ask the network to produce the whole thing, and score
it against the original. It has to infer the missing region from the context,
which is a harder version of denoising. This is the same task Photoshop's
content-aware fill does, and it is also what a diffusion model does when you
use it for inpainting next week.
:::

## Anomaly detection {.smaller}

[![](images/ae-anomaly-flow.png){.dw68}](images/ae-anomaly-flow.png){target="_blank" .zoom}

::: {.notes}
Train on normal data only. The model gets good at rebuilding the kind of thing
it saw and stays bad at everything else, so reconstruction error becomes a
score for "unlike the training data". This is the most shippable thing in
today's lecture, and the reason is that it needs no labels for the anomalies,
which in a real anomaly problem you do not have.
:::

## Where to put the threshold {.smaller}

[![](images/ae-anomaly.png){.dw58}](images/ae-anomaly.png){target="_blank" .zoom}

::: {.notes}
The errors on normal data form a distribution, and the cut goes out in its
tail. Picking the cut is picking your false-alarm rate, and that is a business
decision rather than a modeling one: how many good transactions are you
willing to stop in order to catch one bad one. Note the threshold is set
without ever looking at an anomaly.
:::

## Does it actually work {.smaller}

[![](images/fig-anomaly.png){.dw95}](images/fig-anomaly.png){target="_blank" .zoom}

A 16-unit bottleneck trained on MNIST for 20 epochs, then shown Fashion-MNIST.
**95% of the clothes land past the threshold, for 1% false alarms.**

::: {.notes}
The threshold is the 99th percentile of the training errors. It never sees a
single anomaly, which is the point: at deployment you do not have them either.
Fraud, machine failure, bad sensor readings, and manufacturing defects are all
this shape.
:::

## What generation would require {.smaller}

[![](images/ae-generative.png){.dw70}](images/ae-generative.png){target="_blank" .zoom}

Draw a code from a distribution you can sample, decode it, get something new.

::: {.notes}
Notice what this picture assumes. There is a distribution on the left you can
draw from, and every draw decodes to something plausible. An autoencoder gives
you a decoder, so it looks like it should work. The next three slides are why
it does not.
:::

## The latent space has no required shape {.smaller}

[![](images/vae-structure.png){.dw85}](images/vae-structure.png){target="_blank" .zoom}

::: {.notes}
The training objective says rebuild the input. It says nothing about where the
codes go. Any of those three layouts gets the same reconstruction loss, so the
encoder is free to pick whichever one is convenient, and it will usually leave
large empty regions between the clusters.
:::

## So you cannot sample from it {.smaller}

Decoding a point the encoder never produced usually gives nonsense.

[![](images/vae-sampling.png){.dw85}](images/vae-sampling.png){target="_blank" .zoom}

::: {.notes}
Pick a point between the cat cluster and the dog cluster and decode it. Nothing
in training required that point to mean anything, and you get the hybrid. The
problem is not that the decoder is weak. It is that nobody ever asked the
space to be continuous.
:::

## A VAE encodes a region {.smaller}

A variational autoencoder's encoder outputs a mean and a spread instead of a
single point, and a fresh code is drawn from that region on every pass.

[![](images/fig-ae-vs-vae.png){.dw100}](images/fig-ae-vs-vae.png){target="_blank" .zoom}

::: {.notes}
Because the code moves every pass, nearby codes have to decode to similar
things. That is what fills in the gaps. The decoder is no longer memorizing a
lookup from a handful of points; it has to make the whole neighborhood work.
:::

## The loss function {.smaller}

[![](images/vae-lossfn.png){.dw68}](images/vae-lossfn.png){target="_blank" .zoom}

::: {.notes}
Two terms on a balance. Reconstruction is the autoencoder loss you already
know, and it pulls codes apart so inputs stay distinguishable. The KL term
measures how far the encoder's output distribution sits from a standard
normal and pulls everything back toward it.

Read the second term carefully, because it names a specific target
distribution. It is not asking for a tidy latent space in the abstract. It is
asking for one particular distribution, the one you will later draw from.
That is what makes sampling possible, and the next slide is what it buys.
:::

## What the KL term buys {.smaller}

[![](images/vae-loss.png){.dw80}](images/vae-loss.png){target="_blank" .zoom}

- **Continuity.** The points *between* two codes decode to something sensible
- **A known place to draw from.** The codes pile up on $N(0, I)$

::: {.notes}
Two separate jobs, and you need both. Continuity is the local one, and it is
what the handwritten notes on the left panel are about: a point that decodes
to nothing, and two nearby points that decode to unrelated things.

The second is global and is easy to skip past. Pulling the codes onto a
standard normal tells you *where the codes live*. A latent space could be
perfectly continuous and still be useless to sample, if its codes sat in a
blob centred at four hundred with a spread of fifty, because you would have
no idea what distribution to draw from. The KL term fixes the address, not
just the smoothness.

Run reconstruction alone and you get the left panel. Run KL alone and
everything collapses onto the prior and the decoder learns nothing.
:::

## The reparameterization trick {.smaller}

[![](images/vae-reparam.png){.dw80}](images/vae-reparam.png){target="_blank" .zoom}

::: {.notes}
Drawing a sample is not a differentiable operation, so a gradient cannot pass
through it. Writing the draw as mu plus sigma times noise moves the randomness
into an input, and the path from the loss back to mu and sigma is ordinary
arithmetic again. This is a mechanism rather than a decision you will make,
but it is a standard interview question.
:::

## The same bottleneck, both losses {.smaller}

[![](images/fig-latent.png){.dw92}](images/fig-latent.png){target="_blank" .zoom}

Two units, 20 epochs, same data. **The autoencoder spreads to 28;
the VAE stays inside 5.1.**

::: {.notes}
Look at the axes before the colors. The autoencoder has spokes and empty
wedges, and no reason to prefer any particular scale. The VAE is a single blob
around the origin, because that is what the KL term asked for. Now a random
draw from a standard normal lands somewhere the decoder understands.
:::

## Walking the latent space {.smaller}

[![](images/fig-manifold.png){.dh460}](images/fig-manifold.png){target="_blank" .zoom}

::: {.notes}
A grid of 324 points, none of which came from an image. Both halves of
the KL term's job are visible here, and it is worth separating them.

Continuity is why the squares *between* the digits are digits at all, and why
7 slides into 9 and 3 into 8 instead of going through noise.

The axis range is the other half. This grid runs from minus 2.2 to
plus 2.2, and it is full, because the codes were pulled onto a
standard normal and that square covers almost all of them. On the plain
autoencoder from the last slide the codes ran out past 28, so the same square
would have been nearly empty. Continuity tells you the space is smooth. The
prior tells you where to look.
:::

## The axes can mean something {.smaller}

[![](images/vae-axes.png){.dw68}](images/vae-axes.png){target="_blank" .zoom}

::: {.notes}
Move one latent coordinate and hold the rest still, and sometimes a single
human-readable property changes: glasses, hair colour, pose. Nothing in the
loss asked for that. It happens because the KL term pushes the coordinates to
be independent of one another, and independent coordinates tend to line up
with independent factors in the data.
:::

## How much structure to ask for {.smaller}

[![](images/vae-beta.png){.dw75}](images/vae-beta.png){target="_blank" .zoom}

A weight on the KL term. **Too small and the gaps come back. Too large and
every input decodes to the same thing.**

::: {.notes}
This is the beta-VAE, and beta is the only hyperparameter of the method worth
knowing. The right-hand panel is the failure people hit in practice and
misread as a bug: the model has satisfied the KL term perfectly by ignoring
the input, and the reconstruction loss is the only thing objecting.
:::

## Why VAE images look soft {.smaller}

:::: {.columns}
::: {.column width="47%"}
**VAE**

[![](images/vae-blurry.png){.dw100}](images/vae-blurry.png){target="_blank" .zoom}
:::
::: {.column width="53%"}
**GAN**

[![](images/gan-sharp.png){.dw100}](images/gan-sharp.png){target="_blank" .zoom}
:::
::::

Squared error on pixels is minimized by averaging over everything the code
could have meant.

::: {.notes}
The blur is the loss function, not a bug. Given an ambiguous code, the safest
prediction under squared error is the mean of all the plausible images, and
the mean of many faces is a smooth face. Next week is about two ways around
this.
:::

## Wrap-up {.smaller}

- An autoencoder compresses with no labels. The bottleneck is the design
- **Nonlinear PCA** is the honest one-line description
- **Anomaly detection** is the job you can ship: train on normal, threshold the error
- A plain latent space has holes, so you cannot sample from it
- A **VAE encodes a region**, so the gaps have to decode
- The KL term does two jobs: **continuity**, and **a known place to draw from**
- **Beta** sets how hard the space is pushed toward a normal
- The blur is the squared-error loss, not a bug

::: {.notes}
Next week: GANs make the loss adversarial instead of squared, and diffusion
generates by undoing noise one step at a time. The VAE does not disappear. It
becomes the compressor that makes diffusion affordable.
:::

## Supplemental content {.smaller}

Optional. Prof. Hickman's [centralized lecture
content](https://jfh.georgetown.domains/centralized-lecture-content/), which is
where most of today's figures come from:

- [Autoencoders](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/deep-learning/auto-encoders/vanilla-auto-encoders/notes.html)
- [Variational autoencoders](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/deep-learning/auto-encoders/variational-auto-encoders/VAE.pdf)
- Runnable demos in the same tree: AE on MNIST, a convolutional AE, a
  denoising AE, and a VAE with a two-unit latent space
- [Overview of generative image modeling](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/computer-vision/generative-image-modeling/overview-of-generative-image-modeling/notes.html),
  which is next week

Also optional: Kingma and Welling, [Auto-Encoding Variational
Bayes](https://arxiv.org/abs/1312.6114) (2013), the VAE paper, and Higgins et
al., [beta-VAE](https://openreview.net/forum?id=Sy2fzU9gl) (2017).

## Lab 10 and Quiz 10 {.smaller}

**Lab 10** is anomaly detection with an autoencoder, then a two-unit VAE and a
walk across its latent space. No labels for the anomalies and no long training
run. Due Wednesday, November 11.

Next week is **Generative Modeling II**: GANs, diffusion, and how a prompt
becomes an image.